### Estimating Dipole Moment of SiC5

Using the pyscf and pyberny packages. Code generated by Gemini Pro 3, edited and commented by BAM.

In [1]:
import pyscf
from pyscf import dft
from pyscf.geomopt.berny_solver import optimize 
from pyscf.hessian import thermo

In [9]:
#Define the SiC5 molecule
mol = pyscf.M(
    atom='''
    Si  0.000  0.0  0.000
    C   0.000  0.0  1.693
    C   0.000  0.0  2.971
    C   0.000  0.0  4.291
    C   0.000  0.0  5.569
    C   0.000  0.0  6.833
    ''',
    #basis='6-311++g(d,p)',
    basis='aug-cc-pVTZ',
    spin=2,
    charge=0,
    unit='Angstrom'
)

In [10]:
#Build the Unrestricted DFT object (UKS) and set the functional
mf = dft.UKS(mol)
mf.xc = 'm06-2x'

In [11]:
#Perform the Geometry Optimization
print("--- Starting Geometry Optimization (M06-2X / aug-cc-pVTZ) ---")
mol_eq = optimize(mf)

--- Starting Geometry Optimization (M06-2X / 6-311++G(d,p)) ---

Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
  Si   0.000000   0.000000   0.000000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   1.693000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   2.971000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   4.291000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   5.569000    0.000000  0.000000  0.000000
   C   0.000000   0.000000   6.833000    0.000000  0.000000  0.000000

WARN: Mole.unit (Angstrom) is changed to Bohr

converged SCF energy = -479.703839550326  <S^2> = 2.0901146  2S+1 = 3.0594866
--------------- UKS_Scanner gradients ---------------
         x                y                z
0 Si     0.0000000000    -0.0000000000     0.0153803739
1 C    -0.0000000000    -0.0000000000    -0.0236429535
2 C    -0.0000000000     0.0000000000    -0.04052138

In [12]:
print("\nOptimized Geometry Structure:")
print(mol_eq.tostring())


Optimized Geometry Structure:
Si          0.00000000        0.00000000       -0.00337658
C           0.00000000        0.00000000        1.71222562
C           0.00000000        0.00000000        2.98698174
C           0.00000000        0.00000000        4.26731326
C           0.00000000        0.00000000        5.55212712
C           0.00000000        0.00000000        6.84172884


In [13]:
#Run a final single-point calculation on the optimized structure.
# 'optimize()' returns a new Mole object (mol_eq) with updated coordinates.
print("\n--- Running Final Calculation on Optimized Geometry ---")
mf_eq = dft.UKS(mol_eq)
mf_eq.xc = 'm06-2x'
mf_eq.kernel()


--- Running Final Calculation on Optimized Geometry ---

WARN: 1 small eigenvectors of overlap matrix removed because of linear dependency between AOs.


converged SCF energy = -479.706937691061  <S^2> = 2.0955432  2S+1 = 3.0630333


-479.70693769106083

In [14]:
#Extract and print the final dipole moment
print("\n--- Final Dipole Moment Results ---")
dipole_vector = mf_eq.dip_moment()


--- Final Dipole Moment Results ---
Dipole moment(X, Y, Z, Debye): -0.00000, -0.00000, -6.61285


In [15]:
#Compute Rotational Constants for comparison to lab work (B = 921 from McCarthy:2000:766)
print("\n=== ROTATIONAL CONSTANT ===")
# PySCF stores coordinates in Bohr internally; thermo.rotation_const expects Bohr and AMU
masses = mol_eq.atom_mass_list()
coords = mol_eq.atom_coords()

# Calculate the constants in MHz
rot_constants = thermo.rotation_const(masses, coords, unit='GHz')*1000.
print(f'A: {rot_constants[0]:.2f} MHz\nB: {rot_constants[1]:.2f} MHz\nC: {rot_constants[2]:.2f} MHz')


=== ROTATIONAL CONSTANT ===
A: inf MHz
B: 925.05 MHz
C: 925.05 MHz


Given that these are B_e values, this is close enough to the McCarthy work (921 MHz) for the dipole to be a reasonable estimate.